<a href="https://colab.research.google.com/github/rakshitabadiger12-gif/RITUAL/blob/main/BIOREFINE_AI_HackFest26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

%pip -q install gradio plotly pandas


In [2]:

import gradio as gr
import pandas as pd
import plotly.express as px

def run_simulation(
    bagasse_kg,
    moisture,
    allocation,
    temperature,
    ph,
    pathway
):
    # Basic biomass calculations
    allocated = bagasse_kg * allocation / 100
    unallocated = bagasse_kg - allocated
    dry_matter = bagasse_kg * (1 - moisture / 100)

    # Illustrative software-test limits only
    warnings = []

    if temperature < 25 or temperature > 45:
        warnings.append(
            "Temperature is outside the demo test range."
        )

    if ph < 5.5 or ph > 7.5:
        warnings.append(
            "pH is outside the demo test range."
        )

    # Placeholder coefficients for UI testing only.
    # These are NOT validated industrial yields.
    demo_factors = {
        "Bioethanol": 0.20,
        "Biogas": 0.35,
        "Biofertilizer": 0.70,
        "Bioplastics": 0.15
    }

    estimated_output = (
        allocated * demo_factors[pathway]
    )

    # Status panel
    if warnings:
        status = (
            "### Potential process deviation\n"
            + "\n".join(f"- {w}" for w in warnings)
        )
    else:
        status = (
            "### No deviation detected by demo rules\n"
            "This is a rule-based result, not an AI prediction."
        )

    # KPI report
    report = pd.DataFrame({
        "KPI": [
            "Bagasse input (kg)",
            "Dry matter estimate (kg)",
            "Allocated feedstock (kg)",
            "Unallocated feedstock (kg)",
            "Feedstock utilization (%)",
            "Estimated product output (kg)",
            "Mass-balance error (kg)"
        ],
        "Value": [
            bagasse_kg,
            dry_matter,
            allocated,
            unallocated,
            allocation,
            estimated_output,
            bagasse_kg - allocated - unallocated
        ]
    })

    # Material allocation chart
    chart_data = pd.DataFrame({
        "Category": ["Allocated", "Unallocated"],
        "Mass (kg)": [allocated, unallocated]
    })

    fig = px.bar(
        chart_data,
        x="Category",
        y="Mass (kg)",
        title="BIOREFINE AI: Material Allocation",
        text="Mass (kg)"
    )

    fig.update_layout(
        template="plotly_white",
        showlegend=False
    )

    return status, fig, report


with gr.Blocks(
    title="BIOREFINE AI",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        """
        # 🌱 BIOREFINE AI
        ### AI-powered biorefinery decision-support prototype

        Simulate sugarcane-bagasse allocation, explore
        potential product pathways and inspect process alerts.

        **Prototype v0.1:** product coefficients are illustrative.
        Process alerts currently use predefined rules.
        """
    )

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Feedstock parameters")

            bagasse = gr.Number(
                label="Bagasse input (kg)",
                value=100,
                minimum=1
            )

            moisture = gr.Slider(
                0, 100,
                value=30,
                step=1,
                label="Feedstock moisture (%)"
            )

            allocation = gr.Slider(
                0, 100,
                value=70,
                step=5,
                label="Feedstock allocation (%)"
            )

        with gr.Column():
            gr.Markdown("### Process monitoring")

            temperature = gr.Slider(
                10, 100,
                value=35,
                step=1,
                label="Test temperature (°C)"
            )

            ph = gr.Slider(
                0, 14,
                value=6.5,
                step=0.1,
                label="Test pH"
            )

            pathway = gr.Dropdown(
                choices=[
                    "Bioethanol",
                    "Biogas",
                    "Biofertilizer",
                    "Bioplastics"
                ],
                value="Bioethanol",
                label="Product pathway"
            )

    run_button = gr.Button(
        "Run Biorefinery Simulation",
        variant="primary"
    )

    status = gr.Markdown()

    chart = gr.Plot(label="Material allocation")

    report = gr.Dataframe(
        label="KPI report",
        interactive=False
    )

    run_button.click(
        fn=run_simulation,
        inputs=[
            bagasse,
            moisture,
            allocation,
            temperature,
            ph,
            pathway
        ],
        outputs=[status, chart, report]
    )

demo.launch(share=True)


/tmp/ipykernel_3367/1816627858.py:100: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://60627522a62dd0b7a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:

%pip -q install scikit-learn


In [4]:

import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)
from sklearn.model_selection import train_test_split

# Reproducible synthetic sensor data.
# This is demonstration data, not factory data.
rng = np.random.default_rng(42)

normal = pd.DataFrame({
    "temperature": rng.normal(35, 3, 1200),
    "ph": rng.normal(6.5, 0.3, 1200),
    "moisture": rng.normal(50, 8, 1200)
})

# Split normal observations into training and test sets.
train_data, test_normal = train_test_split(
    normal,
    test_size=0.30,
    random_state=42
)

# Generate separate synthetic abnormal test cases.
abnormal = pd.DataFrame({
    "temperature": rng.choice([10, 80], 200),
    "ph": rng.choice([3.0, 10.0], 200),
    "moisture": rng.choice([5, 95], 200)
})

# Use the same feature order everywhere.
features = ["temperature", "ph", "moisture"]

# Fit the model only on training data.
model = IsolationForest(
    n_estimators=150,
    contamination="auto",
    random_state=42
)

model.fit(train_data[features])

# Evaluate on unseen normal and abnormal test cases.
test_data = pd.concat(
    [test_normal, abnormal],
    ignore_index=True
)

y_true = np.concatenate([
    np.zeros(len(test_normal), dtype=int),
    np.ones(len(abnormal), dtype=int)
])

# IsolationForest: -1 = anomaly, 1 = normal.
predictions = model.predict(test_data[features])
y_pred = (predictions == -1).astype(int)

print("Synthetic test-set results")
print("---------------------------")
print("Test observations:", len(test_data))
print("Actual anomalies:", int(y_true.sum()))
print("Detected anomalies:", int(y_pred.sum()))
print("Precision:", round(
    precision_score(y_true, y_pred, zero_division=0), 3
))
print("Recall:", round(
    recall_score(y_true, y_pred, zero_division=0), 3
))
print("F1-score:", round(
    f1_score(y_true, y_pred, zero_division=0), 3
))

print(
    "\nNote: these metrics evaluate synthetic test data only. "
    "They do not establish real industrial performance."
)


Synthetic test-set results
---------------------------
Test observations: 560
Actual anomalies: 200
Detected anomalies: 268
Precision: 0.746
Recall: 1.0
F1-score: 0.855

Note: these metrics evaluate synthetic test data only. They do not establish real industrial performance.


In [6]:

test_reading = pd.DataFrame([{
    "temperature": 35.0,
    "ph": 6.5,
    "moisture": 50.0
}])

prediction = model.predict(test_reading[features])[0]
score = model.decision_function(test_reading[features])[0]

if prediction == -1:
    print("AI RESULT: Potential anomaly detected")
else:
    print("AI RESULT: No anomaly detected")

print("Anomaly score:", round(float(score), 4))
print("Test reading:")
print(test_reading.to_string(index=False))


AI RESULT: No anomaly detected
Anomaly score: 0.1186
Test reading:
 temperature  ph  moisture
        35.0 6.5      50.0


In [7]:

import gradio as gr
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

# ==========================================
# 1. TRAIN A LOCAL AI ANOMALY DETECTOR
# ==========================================
rng = np.random.default_rng(42)

features = ["temperature", "ph", "moisture"]

normal_data = pd.DataFrame({
    "temperature": rng.normal(35, 3, 1500),
    "ph": rng.normal(6.5, 0.3, 1500),
    "moisture": rng.normal(50, 8, 1500)
})

# Train only on the training portion of normal data.
train_data, test_normal = train_test_split(
    normal_data,
    test_size=0.30,
    random_state=42
)

model = IsolationForest(
    n_estimators=150,
    contamination="auto",
    random_state=42
)

model.fit(train_data[features])

# Evaluate using separate synthetic test examples.
abnormal_data = pd.DataFrame({
    "temperature": rng.choice([10, 80], 200),
    "ph": rng.choice([3.0, 10.0], 200),
    "moisture": rng.choice([5, 95], 200)
})

test_data = pd.concat(
    [test_normal, abnormal_data],
    ignore_index=True
)

y_true = np.concatenate([
    np.zeros(len(test_normal), dtype=int),
    np.ones(len(abnormal_data), dtype=int)
])

y_pred = (
    model.predict(test_data[features]) == -1
).astype(int)

precision = precision_score(
    y_true, y_pred, zero_division=0
)
recall = recall_score(
    y_true, y_pred, zero_division=0
)
f1 = f1_score(
    y_true, y_pred, zero_division=0
)

print("AI model trained successfully.")
print("Synthetic test-set results:")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")

# ==========================================
# 2. SIMULATION AND AI FUNCTION
# ==========================================
def run_simulation(
    bagasse_kg,
    feed_moisture,
    allocation,
    temperature,
    ph,
    pathway
):
    if bagasse_kg <= 0:
        raise gr.Error("Bagasse input must be greater than zero.")

    allocated = bagasse_kg * allocation / 100
    unallocated = bagasse_kg - allocated

    dry_matter = bagasse_kg * (
        1 - feed_moisture / 100
    )

    # Illustrative placeholder coefficients.
    # Not validated industrial yields.
    factors = {
        "Bioethanol": 0.20,
        "Biogas": 0.35,
        "Biofertilizer": 0.70,
        "Bioplastics": 0.15
    }

    estimated_output = allocated * factors[pathway]

    # Use the same features and order as model training.
    sensor_reading = pd.DataFrame([{
        "temperature": temperature,
        "ph": ph,
        "moisture": feed_moisture
    }])[features]

    ai_prediction = model.predict(sensor_reading)[0]
    anomaly_score = float(
        model.decision_function(sensor_reading)[0]
    )

    # Rule-based checks are independent of AI prediction.
    warnings = []

    if temperature < 25 or temperature > 45:
        warnings.append(
            "Temperature is outside the illustrative demo range."
        )

    if ph < 5.5 or ph > 7.5:
        warnings.append(
            "pH is outside the illustrative demo range."
        )

    if feed_moisture < 20 or feed_moisture > 80:
        warnings.append(
            "Moisture is outside the illustrative demo range."
        )

    if ai_prediction == -1:
        ai_status = (
            "### AI: Potential anomaly detected\n"
            "The sensor pattern was flagged by the model."
        )
    else:
        ai_status = (
            "### AI: No anomaly detected\n"
            "The model did not flag this sensor pattern."
        )

    ai_status += (
        f"\n\n**Anomaly score:** `{anomaly_score:.4f}`"
        "\n\nThis score is not a failure probability."
    )

    if warnings:
        rule_status = (
            "### Rule-based warnings\n"
            + "\n".join(f"- {w}" for w in warnings)
        )
    else:
        rule_status = (
            "### Rule-based checks passed\n"
            "No configured demo limits were exceeded."
        )

    # Material balance chart.
    chart_data = pd.DataFrame({
        "Category": ["Allocated", "Unallocated"],
        "Mass (kg)": [allocated, unallocated]
    })

    fig = px.bar(
        chart_data,
        x="Category",
        y="Mass (kg)",
        text="Mass (kg)",
        title="Bagasse Material Allocation"
    )

    fig.update_layout(
        template="plotly_white",
        showlegend=False
    )

    # Scenario report.
    report = pd.DataFrame({
        "KPI": [
            "Bagasse input (kg)",
            "Dry matter estimate (kg)",
            "Allocated feedstock (kg)",
            "Unallocated feedstock (kg)",
            "Feedstock utilization (%)",
            "Estimated product output (kg)",
            "Temperature (C)",
            "pH",
            "Feedstock moisture (%)",
            "AI prediction",
            "AI anomaly score",
            "Mass balance error (kg)"
        ],
        "Value": [
            bagasse_kg,
            dry_matter,
            allocated,
            unallocated,
            allocation,
            estimated_output,
            temperature,
            ph,
            feed_moisture,
            "Anomaly" if ai_prediction == -1 else "Normal",
            anomaly_score,
            bagasse_kg - allocated - unallocated
        ]
    })

    evaluation = (
        "### AI model evaluation\n"
        "**Training data:** synthetic normal sensor readings\n\n"
        f"- Precision: {precision:.3f}\n"
        f"- Recall: {recall:.3f}\n"
        f"- F1-score: {f1:.3f}\n\n"
        "These metrics describe a synthetic test set, "
        "not validated industrial performance."
    )

    return ai_status, rule_status, fig, report, evaluation


# ==========================================
# 3. USER INTERFACE
# ==========================================
with gr.Blocks(
    title="BIOREFINE AI",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown(
        """
        # 🌱 BIOREFINE AI
        ### AI-powered biorefinery digital-twin prototype

        Explore sugarcane-bagasse allocation, monitor
        sensor patterns and compare simulated outcomes.

        **HackFest 2026 | Synthetic AI training data**
        """
    )

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Feedstock parameters")

            bagasse = gr.Number(
                label="Bagasse input (kg)",
                value=100,
                minimum=1
            )

            feed_moisture = gr.Slider(
                minimum=0,
                maximum=100,
                value=50,
                step=1,
                label="Feedstock moisture (%)"
            )

            allocation = gr.Slider(
                minimum=0,
                maximum=100,
                value=70,
                step=5,
                label="Feedstock allocation (%)"
            )

        with gr.Column():
            gr.Markdown("### Process monitoring")

            temperature = gr.Slider(
                minimum=10,
                maximum=100,
                value=35,
                step=1,
                label="Temperature (°C)"
            )

            ph = gr.Slider(
                minimum=0,
                maximum=14,
                value=6.5,
                step=0.1,
                label="pH"
            )

            pathway = gr.Dropdown(
                choices=[
                    "Bioethanol",
                    "Biogas",
                    "Biofertilizer",
                    "Bioplastics"
                ],
                value="Bioethanol",
                label="Product pathway"
            )

    run_button = gr.Button(
        "Run Simulation + AI Analysis",
        variant="primary"
    )

    ai_output = gr.Markdown()
    rule_output = gr.Markdown()

    chart_output = gr.Plot(
        label="Digital Twin: Material Allocation"
    )

    report_output = gr.Dataframe(
        label="KPI and Scenario Report",
        interactive=False
    )

    evaluation_output = gr.Markdown()

    run_button.click(
        fn=run_simulation,
        inputs=[
            bagasse,
            feed_moisture,
            allocation,
            temperature,
            ph,
            pathway
        ],
        outputs=[
            ai_output,
            rule_output,
            chart_output,
            report_output,
            evaluation_output
        ]
    )

    gr.Markdown(
        """
        ---
        **Prototype limitations:** Outputs use illustrative
        yield coefficients and synthetic training data.
        Alerts do not establish actual industrial safety.
        """
    )

demo.launch(share=True)


AI model trained successfully.
Synthetic test-set results:
Precision: 0.694
Recall:    1.000
F1-score:  0.820


/tmp/ipykernel_3367/1177361060.py:231: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://61840750e385135a9a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
